In [1]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B"

# Set up 4-bit quantization config (recommended to save VRAM)
# Set quantization_config=None in from_pretrained if you have a powerful GPU (e.g. RTX 3090/4090) and want to do 16-bit LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

c:\Users\HP\Documents\Coding journeys\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]c:\Users\HP\Documents\Coding journeys\GenAI\venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\HP\Documents\Coding journeys\GenAI\venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [02:12<00:00,  3.28it/s]


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Qwen doesn't have a default pad token, so we map it to the eos_token
tokenizer.pad_token = tokenizer.eos_token

In [4]:
from peft import LoraConfig

config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Qwen 2.5 uses these target modules for optimal attention + MLP projection tuning.
    # If you are highly VRAM-constrained, you can restrict this to ["q_proj", "v_proj"]
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from peft import get_peft_model

model = get_peft_model(model, config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("lora_adapter")

In [ ]:
import torch
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base,
    "lora_adapter"
)

In [ ]:
merged = model.merge_and_unload()

In [ ]:
merged.save_pretrained("merged_model")

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B",
    quantization_config=bnb,
    device_map="auto"
)